# Guía para leer el proyecto desde cero

**Para quién es esta guía.** Para alguien que no ha trabajado con precios, series de tiempo ni
modelos predictivos. Explica qué contiene el dataset, cómo se limpia, cómo se construyen las
variables y cómo se leen los resultados.

**Qué se busca.** Entender si el precio de Lima contiene información útil para estimar el cambio de
precio de los departamentos y si esa información sirve para pronosticar el mes siguiente.

## 1. Qué contiene una fila

Cada fila es un **departamento, una categoría y un mes**:

- `fecha`: primer día del mes;
- `departamento_id`: código ISO del departamento;
- `departamento`: nombre oficial;
- `region`: alias ASCII usado por el código;
- `categoria`: producto armonizado;
- `precio_kg`: precio promedio en soles por kilogramo;
- `n_registros`: cuántos precios se promediaron.

La fuente tiene **variedades comerciales**. El panel elige una variedad y una unidad dominante por
departamento y producto, y armoniza todo en 9 categorías.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd()
if not (RAIZ / "data").exists():
    RAIZ = RAIZ.parent
TAB = RAIZ / "outputs" / "informe" / "tablas"
AN = RAIZ / "outputs" / "analisis"

panel = pd.read_csv(RAIZ / "data" / "processed" / "panel_precios_limpio.csv", parse_dates=["fecha"])
display(panel.head(8))
print(f"{len(panel):,} filas | {panel.region.nunique()} departamentos | {panel.categoria.nunique()} productos")

,fecha,anio,mes,departamento_id,departamento,region,categoria,n_registros,precio_kg
0,2008-09-01,2008,9,PE-AMA,Amazonas,Amazonas,Aceite,1,5.56
1,2008-10-01,2008,10,PE-AMA,Amazonas,Amazonas,Aceite,1,6.39
2,2008-11-01,2008,11,PE-AMA,Amazonas,Amazonas,Aceite,1,6.72
3,2008-12-01,2008,12,PE-AMA,Amazonas,Amazonas,Aceite,1,6.64
4,2009-01-01,2009,1,PE-AMA,Amazonas,Amazonas,Aceite,1,6.34
5,2009-02-01,2009,2,PE-AMA,Amazonas,Amazonas,Aceite,1,5.71
6,2009-03-01,2009,3,PE-AMA,Amazonas,Amazonas,Aceite,1,5.54
7,2009-04-01,2009,4,PE-AMA,Amazonas,Amazonas,Aceite,1,5.66


43,855 filas | 24 departamentos | 9 productos


## 2. Unidad de análisis: 24 departamentos

El catálogo `data/reference/departamentos.csv` es la fuente única de nombres, códigos y
coordenadas.

SISAP también publica Andahuaylas, Chota, Jaén y Callao. No se incluyen:

- Andahuaylas es provincia de Apurímac;
- Chota y Jaén son provincias de Cajamarca;
- Callao es provincia constitucional.

Incluirlos como departamentos duplicaría la unidad y rompería la comparación Lima–Puno.

## 3. Limpieza

Se revisan nulos, duplicados, unidades y atípicos.

- Las unidades se normalizan a kilogramos.
- Un valor se marca como atípico si se aleja demasiado de la mediana local o nacional.
- No se imputan valores: se excluyen y se reportan.

La tabla muestra cuántas filas había antes y después, y cuántos atípicos se descartaron.

In [2]:
display(pd.read_csv(TAB / "tabla_02_auditoria.csv"))
display(pd.read_csv(TAB / "tabla_22_verificacion_limpieza.csv"))

,metrica,valor
0,filas,43921
1,departamentos,24
2,productos,9
3,nulos,0
4,duplicados fecha-departamento-producto,0
5,atipicos detectados,66
6,atipicos (%),0.15
7,primer mes,1997-01
8,ultimo mes,2026-11
9,cobertura 2012-2023 bruta (%),85.9


,metrica,valor
0,filas antes,43921.0
1,filas despues,43855.0
2,filas descartadas,66.0
3,atipicos marcados,66.0
4,nulos antes,0.0
5,nulos despues,0.0
6,duplicados antes,0.0
7,duplicados despues,0.0
8,cobertura 2012-2023 antes (%),85.9
9,cobertura 2012-2023 despues (%),85.7


![](../outputs/informe/figuras/23_verificacion_limpieza.png)

## 4. Cobertura

El panel no es balanceado: no todos los departamentos tienen todos los productos todos los meses.
La cobertura se calcula contra la grilla departamento-categoría-mes.

En 2000 hay 22 de los 24 departamentos y 7 categorías; la cobertura densa empieza en 2012.

![](../outputs/informe/figuras/02_cobertura.png)

## 5. Ventanas temporales y `N`

| Ventana | Uso |
|---|---|
| 1997–2026 | Fuente completa |
| 2012–2023 | EDA y ventana densa |
| 2012–2017 | Entrenamiento |
| 2018–2020 | Validación |
| 2021–octubre 2026 | Prueba final |
| Noviembre 2026 | Fuera de la prueba |

El `N` no es el número de meses calendario: es el número de meses con datos completos después de
construir cambios y rezagos. Por eso cambia entre productos y entre tareas.

In [3]:
display(pd.read_csv(AN / "06_modelos" / "n_etapas.csv"))

,producto,tarea,entrenamiento_2012_2017,validacion_2018_2020,prueba_2021_2026,fuera_de_ventana,total
0,Papa,estimación del mes corriente,67,28,60,1,156
1,Papa,pronóstico a un mes,69,29,62,0,160
2,Cebolla,estimación del mes corriente,67,17,60,1,145
3,Cebolla,pronóstico a un mes,69,19,62,0,150
4,Zanahoria,estimación del mes corriente,52,17,52,0,121
5,Zanahoria,pronóstico a un mes,57,19,53,0,129
6,Arroz,estimación del mes corriente,68,27,31,1,127
7,Arroz,pronóstico a un mes,71,26,36,0,133
8,Aceite,estimación del mes corriente,68,32,24,0,124
9,Aceite,pronóstico a un mes,71,33,24,0,128


## 6. De la tabla a la matriz X/y

Para cada producto se construye una tabla mensual de Puno y Lima.

- `y`: cambio logarítmico de Puno.
- `dl0`: cambio de Lima del mismo mes.
- `dl1`–`dl3`: cambios de Lima rezagados.
- `dp1`–`dp2`: cambios propios rezagados.
- `sin1`, `cos1`: estacionalidad anual.

La matriz se corta por fecha: entrenamiento, validación y prueba final.

In [4]:
display(pd.read_csv(TAB / "tabla_24_matriz_diseno.csv").round(3))

,fecha,producto,region,dl0,dl1,dl2,dl3,dp1,dp2,sin1,cos1,y_cambio_log,split
0,2012-06-01,Papa,Puno,0.000,-0.134,-0.012,-0.211,0.000,-0.101,0.500,-0.866,-0.019,entrenamiento
1,2012-07-01,Papa,Puno,0.108,0.000,-0.134,-0.012,-0.019,0.000,0.000,-1.000,-0.030,entrenamiento
2,2012-08-01,Papa,Puno,0.000,0.108,0.000,-0.134,-0.030,-0.019,-0.500,-0.866,0.040,entrenamiento
3,2012-09-01,Papa,Puno,-0.066,0.000,0.108,0.000,0.040,-0.030,-0.866,-0.500,-0.081,entrenamiento
4,2026-07-01,Papa,Puno,-0.160,-0.211,-0.168,-0.194,-0.164,-0.025,0.000,-1.000,-0.067,holdout
5,2026-08-01,Papa,Puno,0.419,-0.160,-0.211,-0.168,-0.067,-0.164,-0.500,-0.866,-0.038,holdout
6,2026-09-01,Papa,Puno,0.381,0.419,-0.160,-0.211,-0.038,-0.067,-0.866,-0.500,0.217,holdout
7,2026-10-01,Papa,Puno,-0.009,0.381,0.419,-0.160,0.217,-0.038,-1.000,0.000,0.159,holdout


## 7. Cómo se leen las métricas

| Métrica | Qué mide | Mejor si... |
|---|---|---|
| R² | proporción de variación explicada | más alto |
| MAE | error absoluto promedio | más bajo |
| Mejora del MAE | reducción frente a la referencia | positiva |
| Acierto direccional | porcentaje de signos correctos | más alto que 50 % |

La estimación del mes corriente se evalúa sobre **cambios**. El pronóstico a un mes se evalúa sobre
**niveles de precio** y se compara con la caminata aleatoria.

## 8. Modelos y resultados

Se comparan cuatro modelos:

1. OLS: regresión lineal.
2. Ridge: regresión lineal con penalización.
3. Random Forest: muchos árboles que promedian.
4. LightGBM: boosting de árboles.

La selección se hace por rango promedio de R² en validación. La prueba final se usa una sola vez.

In [5]:
display(pd.read_csv(TAB / "tabla_08_torneo.csv").round(3))
display(pd.read_csv(TAB / "tabla_09_ganador.csv").query("periodo == 'holdout' and modelo == 'Random Forest'")[["producto", "r2", "acierto", "n"]].round(3))

,modelo,rank_promedio,r2_medio,r2_mediano,acierto_medio,productos_rankeados
0,Random Forest,1.50,-0.687,-0.097,46.879,4
1,Ridge,1.75,-0.661,-0.137,46.228,4
2,OLS,2.75,-0.689,-0.153,46.228,4
3,LightGBM,4.00,-2.434,-0.544,42.785,4


,producto,r2,acierto,n
23,Papa,0.270,69.492,60
28,Cebolla,0.218,64.407,60
33,Zanahoria,0.080,52.941,52
38,Arroz,-0.842,41.935,31
43,Aceite,-0.103,50.000,24
48,Huevos,-0.013,55.263,38


![](../outputs/informe/figuras/11_ranking_torneo.png)

![](../outputs/informe/figuras/12_ganador_vs_observado.png)

## 9. Pronóstico a un mes

La tabla muestra la mejora porcentual del MAE frente a la caminata aleatoria. Positivo = mejora;
negativo = empeora.

In [6]:
display(pd.read_csv(TAB / "tabla_23_pronostico_holdout.csv").round(1))

,producto,n,OLS,Ridge,Random Forest,LightGBM
0,Papa,62,14.5,12.7,-74.2,-85.6
1,Cebolla,62,8.3,9.9,-2.8,-12.5
2,Zanahoria,53,-4.7,-7.7,-62.0,-59.8
3,Arroz,36,3.9,7.7,-282.0,-353.2
4,Aceite,24,-37.8,-36.6,-303.1,-306.8
5,Huevos,38,-3.6,-3.3,-146.4,-142.3


## 10. Dirección y matriz de confusión

Se clasifica el signo del cambio observado y del predicho. Los meses con cambio exactamente cero se
excluyen porque no tienen dirección.

In [7]:
display(pd.read_csv(TAB / "tabla_25_confusion_modelos.csv").round(1))

,modelo,TP,FN,FP,TN,n,acierto
0,LightGBM,97,53,46,66,262,62.2
1,OLS,99,51,56,56,262,59.2
2,Random Forest,92,58,52,60,262,58.0
3,Ridge,99,51,56,56,262,59.2


![](../outputs/informe/figuras/24_matriz_confusion.png)

## 11. Cómo interpretar un resultado por producto

No todos los productos se comportan igual:

- Papa y cebolla tienen mejor señal contemporánea.
- Zanahoria mejora poco.
- Arroz, aceite y huevos son difíciles de pronosticar.
- Ridge y OLS suelen ser mejores para el pronóstico a un mes.
- Random Forest suele ser mejor para la estimación contemporánea.

Un buen R² en un producto no garantiza buen desempeño en otro.

## 12. Límites

- Los precios son mensuales y no se conoce el rezago de publicación.
- No hay volúmenes, costos de transporte ni precios minoristas.
- El panel no es balanceado.
- Los modelos no demuestran causalidad.
- La tabla fue construida por el equipo: cada decisión de variedad, unidad y limpieza importa.
- El tamaño de muestra es limitado para modelos no lineales.